# RAG-Based AI Report Question Answering System — Version 4

This notebook builds a retrieval-augmented question answering system for long-form AI reports.

## Version 4 Goals

Version 4 extends the earlier retrieval pipeline by adding **evidence-grounded answer generation**.

The full pipeline includes:

1. PDF text extraction  
2. Text chunking with page-level metadata  
3. Noise filtering for Table of Contents, repeated headers, and low-information chunks  
4. Sentence embedding generation  
5. FAISS cosine-similarity retrieval  
6. Cross-encoder reranking  
7. Page-aware retrieval evaluation using Recall@K, Precision@K, and MRR@K  
8. Evidence-context construction  
9. RAG prompt construction  
10. Optional LLM answer generation with source-grounded evidence  
11. Fallback extractive answer generation without external API calls  

## 0. Installation

Run the following cell if your environment does not already have the required packages.

If you are on Windows and `faiss-cpu` has installation issues, you can install it with Conda:

```bash
conda install -c conda-forge faiss-cpu
```

In [ ]:
# Optional: uncomment if needed
# !pip install pymupdf sentence-transformers faiss-cpu pandas numpy tqdm scikit-learn matplotlib python-dotenv openai

## 1. Imports and Configuration

In [ ]:
from pathlib import Path
import os
import re
import json
import warnings

import fitz  # PyMuPDF
import faiss
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from sentence_transformers import SentenceTransformer, CrossEncoder

warnings.filterwarnings("ignore")

# Project paths
PROJECT_ROOT = Path("..")
DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

DATA_RAW_DIR.mkdir(parents=True, exist_ok=True)
DATA_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Update this path if your PDF filename is different
PDF_PATH = DATA_RAW_DIR / "Artificial_ Intelligence_Index_Report_2025.pdf"

SOURCE_NAME = "Stanford AI Index Report 2025"

# Chunking settings
CHUNK_SIZE = 800
CHUNK_OVERLAP = 150

# Retrieval settings
FIRST_K = 20
TOP_K = 5

## 2. Load PDF and Extract Text

This section extracts page-level text from the PDF and stores the page number as metadata.

In [ ]:
def extract_pdf_text(pdf_path):
    """
    Extract page-level text from a PDF.

    Returns:
        list[dict]: Each item contains page number and extracted text.
    """
    pdf_path = Path(pdf_path)

    if not pdf_path.exists():
        raise FileNotFoundError(
            f"PDF not found at: {pdf_path}\n"
            "Please place your PDF in ../data/raw/ or update PDF_PATH."
        )

    doc = fitz.open(pdf_path)
    pages = []

    for page_idx, page in enumerate(doc):
        text = page.get_text("text")
        pages.append({
            "page": page_idx + 1,
            "text": text
        })

    return pages


pages = extract_pdf_text(PDF_PATH)

print("Number of pages:", len(pages))
print("Preview of page 1:")
print(pages[0]["text"][:1000])

In [ ]:
# Check text extraction quality
page_df = pd.DataFrame(pages)
page_df["text_length"] = page_df["text"].apply(len)

display(page_df[["page", "text_length"]].head(20))
display(page_df["text_length"].describe())

## 3. Text Chunking

Long pages are split into overlapping chunks. Each chunk keeps its source page number, which will later be used for citation and evaluation.

In [ ]:
def chunk_text(text, chunk_size=800, overlap=150):
    """
    Split text into overlapping character-based chunks.
    """
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()

        if len(chunk) > 100:
            chunks.append(chunk)

        start += chunk_size - overlap

    return chunks


def build_chunks_dataframe(pages, source_name, chunk_size=800, overlap=150):
    """
    Build a chunk-level dataframe from page-level extracted text.
    """
    all_chunks = []

    for page in pages:
        chunks = chunk_text(
            page["text"],
            chunk_size=chunk_size,
            overlap=overlap
        )

        for chunk_id, chunk in enumerate(chunks):
            all_chunks.append({
                "source": source_name,
                "page": page["page"],
                "chunk_id": chunk_id,
                "text": chunk
            })

    return pd.DataFrame(all_chunks)


chunks_df = build_chunks_dataframe(
    pages=pages,
    source_name=SOURCE_NAME,
    chunk_size=CHUNK_SIZE,
    overlap=CHUNK_OVERLAP
)

print("Total raw chunks:", len(chunks_df))
display(chunks_df.head())

## 4. Clean Noisy Chunks

Version 3 showed that Table of Contents chunks and repeated report headers can cause false positives.  
Version 4 keeps this cleaning stage before building the retrieval index.

In [ ]:
def clean_chunk_text(text):
    """
    Clean repeated headers, page numbers, excessive whitespace, and common PDF artifacts.
    """
    if pd.isna(text):
        return ""

    lines = str(text).split("\n")
    cleaned_lines = []

    noise_patterns = [
        r"^artificial intelligence index report 2025$",
        r"^artificial intelligence index report$",
        r"^artificial intelligence$",
        r"^index report 2025$",
        r"^stanford university$",
        r"^hai$",
        r"^chapter \d+$",
        r"^\d+$"
    ]

    for line in lines:
        line = line.strip()

        if not line:
            continue

        line_lower = line.lower()

        if any(re.fullmatch(pattern, line_lower) for pattern in noise_patterns):
            continue

        cleaned_lines.append(line)

    cleaned_text = " ".join(cleaned_lines)
    cleaned_text = re.sub(r"\s+", " ", cleaned_text).strip()

    return cleaned_text


def is_noisy_chunk(text):
    """
    Identify chunks that are likely not useful for retrieval.
    """
    if pd.isna(text):
        return True

    text = str(text).strip()
    text_lower = text.lower()

    if len(text) < 200:
        return True

    if "table of contents" in text_lower:
        return True

    if text_lower.count("chapter") >= 5 and len(text_lower) < 1200:
        return True

    digit_tokens = re.findall(r"\b\d{1,4}\b", text_lower)
    words = re.findall(r"[a-zA-Z]+", text_lower)

    if len(words) > 0:
        digit_ratio = len(digit_tokens) / len(words)
        if digit_ratio > 0.35:
            return True

    return False


chunks_df_clean = chunks_df.copy()
chunks_df_clean["clean_text"] = chunks_df_clean["text"].apply(clean_chunk_text)
chunks_df_clean["is_noisy"] = chunks_df_clean["clean_text"].apply(is_noisy_chunk)

print("Original chunks:", len(chunks_df_clean))
print("Noisy chunks:", int(chunks_df_clean["is_noisy"].sum()))

chunks_df_clean = chunks_df_clean[~chunks_df_clean["is_noisy"]].copy()
chunks_df_clean = chunks_df_clean.reset_index(drop=True)

print("Clean chunks:", len(chunks_df_clean))

display(chunks_df_clean[["source", "page", "chunk_id", "clean_text"]].head())

In [ ]:
# Save cleaned chunks for reproducibility
chunks_df_clean.to_csv(DATA_PROCESSED_DIR / "chunks_clean_v4.csv", index=False)
print("Saved:", DATA_PROCESSED_DIR / "chunks_clean_v4.csv")

## 5. Build Embeddings and FAISS Cosine Index

For sentence embeddings, cosine similarity is usually more appropriate than raw L2 distance.  
FAISS `IndexFlatIP` with normalized vectors gives cosine similarity.

In [ ]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

clean_texts = chunks_df_clean["clean_text"].tolist()

clean_embeddings = embedding_model.encode(
    clean_texts,
    show_progress_bar=False,
    convert_to_numpy=True
).astype("float32")

print("Clean embedding shape:", clean_embeddings.shape)

# Save embeddings locally. These can be regenerated, so they are usually ignored by Git.
np.save(DATA_PROCESSED_DIR / "clean_embeddings_v4.npy", clean_embeddings)

In [ ]:
def build_faiss_cosine_index(embeddings):
    """
    Build a FAISS index using cosine similarity.
    Inner product on L2-normalized embeddings equals cosine similarity.
    """
    embeddings_cosine = embeddings.copy()
    faiss.normalize_L2(embeddings_cosine)

    dimension = embeddings_cosine.shape[1]
    index = faiss.IndexFlatIP(dimension)
    index.add(embeddings_cosine)

    return index


index_clean_cosine = build_faiss_cosine_index(clean_embeddings)

print("Number of clean vectors in FAISS index:", index_clean_cosine.ntotal)

## 6. Dense Retrieval

In [ ]:
def retrieve_dense_cosine_clean(query, top_k=10):
    """
    Retrieve top-k chunks from the cleaned corpus using cosine similarity.
    """
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        show_progress_bar=False
    ).astype("float32")

    faiss.normalize_L2(query_embedding)

    scores, indices = index_clean_cosine.search(query_embedding, top_k)

    results = []

    for rank, idx in enumerate(indices[0]):
        row = chunks_df_clean.iloc[idx]

        results.append({
            "rank": rank + 1,
            "page": int(row["page"]),
            "chunk_id": int(row["chunk_id"]),
            "cosine_score": float(scores[0][rank]),
            "text": row["clean_text"]
        })

    return results


def show_dense_results(query, top_k=5):
    results = retrieve_dense_cosine_clean(query, top_k=top_k)

    return pd.DataFrame([
        {
            "rank": r["rank"],
            "page": r["page"],
            "chunk_id": r["chunk_id"],
            "cosine_score": round(r["cosine_score"], 4),
            "preview": r["text"][:350]
        }
        for r in results
    ])


show_dense_results("How has AI investment changed in recent years?", top_k=5)

## 7. Cross-Encoder Reranking

Dense retrieval is efficient and works well for candidate generation.  
The cross-encoder reranker then scores each `(query, chunk)` pair more precisely.

In [ ]:
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")


def retrieve_with_rerank_clean(query, first_k=20, top_k=5):
    """
    First retrieve candidate chunks using clean dense retrieval,
    then rerank them with a cross-encoder.
    """
    candidates = retrieve_dense_cosine_clean(query, top_k=first_k)

    pairs = [(query, item["text"]) for item in candidates]
    rerank_scores = reranker.predict(pairs)

    for item, score in zip(candidates, rerank_scores):
        item["rerank_score"] = float(score)

    reranked = sorted(
        candidates,
        key=lambda x: x["rerank_score"],
        reverse=True
    )

    for i, item in enumerate(reranked[:top_k]):
        item["rank"] = i + 1

    return reranked[:top_k]


def show_reranked_results(query, first_k=20, top_k=5):
    results = retrieve_with_rerank_clean(query, first_k=first_k, top_k=top_k)

    return pd.DataFrame([
        {
            "rank": r["rank"],
            "page": r["page"],
            "chunk_id": r["chunk_id"],
            "cosine_score": round(r["cosine_score"], 4),
            "rerank_score": round(r["rerank_score"], 4),
            "preview": r["text"][:350]
        }
        for r in results
    ])


show_reranked_results(
    "How has AI investment changed in recent years?",
    first_k=FIRST_K,
    top_k=TOP_K
)

## 8. Page-Aware Evaluation Benchmark

This benchmark is designed for retrieval evaluation.  
A result is counted as relevant only when it matches both:

1. An expected page range  
2. Expected evidence keywords  

You can refine the `expected_pages` after inspecting the PDF and retrieval results.

In [ ]:
eval_questions_v4 = [
    {
        "question": "How has AI investment changed in recent years?",
        "expected_pages": [3, 18, 260, 364],
        "expected_keywords": ["investment", "private investment"],
        "question_type": "trend"
    },
    {
        "question": "What does the report say about the cost of AI inference?",
        "expected_pages": [2, 5, 13, 65, 100],
        "expected_keywords": ["inference", "cost"],
        "question_type": "trend"
    },
    {
        "question": "What are the trends in AI publications and patents?",
        "expected_pages": [2, 13, 14, 29, 30],
        "expected_keywords": ["publications", "patents"],
        "question_type": "trend"
    },
    {
        "question": "How is AI being used in healthcare?",
        "expected_pages": [167, 298, 310, 318],
        "expected_keywords": ["healthcare", "medical"],
        "question_type": "application"
    },
    {
        "question": "What risks or challenges are associated with AI adoption?",
        "expected_pages": [17, 180, 182, 183, 185],
        "expected_keywords": ["risk", "adoption"],
        "question_type": "risk"
    },
    {
        "question": "What does the report say about foundation models?",
        "expected_pages": [165, 201, 297, 443],
        "expected_keywords": ["foundation model", "model"],
        "question_type": "concept"
    },
    {
        "question": "How are AI models evaluated in the report?",
        "expected_pages": [47, 102, 103, 201],
        "expected_keywords": ["benchmark", "evaluation"],
        "question_type": "method"
    },
    {
        "question": "What does the report say about AI education?",
        "expected_pages": [378, 379, 382, 394],
        "expected_keywords": ["education", "AI"],
        "question_type": "application"
    },
    {
        "question": "What are the trends in responsible AI?",
        "expected_pages": [2, 4, 16, 18, 167],
        "expected_keywords": ["responsible AI", "risk"],
        "question_type": "risk"
    },
    {
        "question": "What does the report say about AI regulation?",
        "expected_pages": [351, 399, 412, 413],
        "expected_keywords": ["regulation", "policy"],
        "question_type": "policy"
    }
]

eval_df_v4 = pd.DataFrame(eval_questions_v4)
display(eval_df_v4)

In [ ]:
def page_hit(result_page, expected_pages, tolerance=1):
    for expected_page in expected_pages:
        if abs(result_page - expected_page) <= tolerance:
            return True
    return False


def keyword_hit_min_matches(text, keywords, min_matches=1):
    text_lower = text.lower()
    matched = []

    for keyword in keywords:
        if keyword.lower() in text_lower:
            matched.append(keyword)

    return len(matched) >= min_matches, matched


def is_relevant_result(
    result,
    expected_pages,
    expected_keywords,
    tolerance=1,
    min_keyword_matches=1
):
    page_ok = page_hit(
        result_page=result["page"],
        expected_pages=expected_pages,
        tolerance=tolerance
    )

    keyword_ok, matched_keywords = keyword_hit_min_matches(
        text=result["text"],
        keywords=expected_keywords,
        min_matches=min_keyword_matches
    )

    return page_ok and keyword_ok, matched_keywords

In [ ]:
def evaluate_recall_at_k_page_keyword(
    eval_df,
    retrieve_func,
    k=5,
    tolerance=1,
    min_keyword_matches=1
):
    hits = 0
    rows = []

    for _, row in eval_df.iterrows():
        query = row["question"]
        expected_pages = row["expected_pages"]
        expected_keywords = row["expected_keywords"]

        results = retrieve_func(query, top_k=k)

        relevant_found = False
        hit_pages = []
        matched_all = []

        for r in results:
            relevant, matched = is_relevant_result(
                r,
                expected_pages=expected_pages,
                expected_keywords=expected_keywords,
                tolerance=tolerance,
                min_keyword_matches=min_keyword_matches
            )

            if relevant:
                relevant_found = True
                hit_pages.append(r["page"])
                matched_all.extend(matched)

        if relevant_found:
            hits += 1

        rows.append({
            "question": query,
            "recall_hit": int(relevant_found),
            "hit_pages": hit_pages,
            "matched_keywords": list(set(matched_all))
        })

    recall = hits / len(eval_df)

    return recall, pd.DataFrame(rows)


def evaluate_precision_at_k_page_keyword(
    eval_df,
    retrieve_func,
    k=5,
    tolerance=1,
    min_keyword_matches=1
):
    precision_scores = []
    rows = []

    for _, row in eval_df.iterrows():
        query = row["question"]
        expected_pages = row["expected_pages"]
        expected_keywords = row["expected_keywords"]

        results = retrieve_func(query, top_k=k)

        relevant_count = 0
        relevant_pages = []

        for r in results:
            relevant, matched = is_relevant_result(
                r,
                expected_pages=expected_pages,
                expected_keywords=expected_keywords,
                tolerance=tolerance,
                min_keyword_matches=min_keyword_matches
            )

            if relevant:
                relevant_count += 1
                relevant_pages.append(r["page"])

        precision = relevant_count / k
        precision_scores.append(precision)

        rows.append({
            "question": query,
            "precision_at_k": precision,
            "relevant_count": relevant_count,
            "relevant_pages": relevant_pages
        })

    mean_precision = sum(precision_scores) / len(precision_scores)

    return mean_precision, pd.DataFrame(rows)


def evaluate_mrr_at_k_page_keyword(
    eval_df,
    retrieve_func,
    k=5,
    tolerance=1,
    min_keyword_matches=1
):
    reciprocal_ranks = []
    rows = []

    for _, row in eval_df.iterrows():
        query = row["question"]
        expected_pages = row["expected_pages"]
        expected_keywords = row["expected_keywords"]

        results = retrieve_func(query, top_k=k)

        first_hit_rank = None
        hit_page = None
        matched_keywords = []
        hit_preview = None

        for i, r in enumerate(results):
            relevant, matched = is_relevant_result(
                r,
                expected_pages=expected_pages,
                expected_keywords=expected_keywords,
                tolerance=tolerance,
                min_keyword_matches=min_keyword_matches
            )

            if relevant:
                first_hit_rank = i + 1
                hit_page = r["page"]
                matched_keywords = matched
                hit_preview = r["text"][:300].replace("\n", " ")
                break

        reciprocal_rank = 0 if first_hit_rank is None else 1 / first_hit_rank
        reciprocal_ranks.append(reciprocal_rank)

        rows.append({
            "question": query,
            "first_hit_rank": first_hit_rank,
            "reciprocal_rank": reciprocal_rank,
            "hit_page": hit_page,
            "matched_keywords": matched_keywords,
            "hit_preview": hit_preview
        })

    mrr = sum(reciprocal_ranks) / len(reciprocal_ranks)

    return mrr, pd.DataFrame(rows)

In [ ]:
def retrieve_clean_dense_wrapper(query, top_k=5):
    return retrieve_dense_cosine_clean(query, top_k=top_k)


def retrieve_clean_rerank_wrapper(query, top_k=5):
    return retrieve_with_rerank_clean(query, first_k=FIRST_K, top_k=top_k)


K = 5
TOLERANCE = 1
MIN_KEYWORD_MATCHES = 1

# Dense retrieval evaluation
dense_recall, dense_recall_detail = evaluate_recall_at_k_page_keyword(
    eval_df_v4,
    retrieve_func=retrieve_clean_dense_wrapper,
    k=K,
    tolerance=TOLERANCE,
    min_keyword_matches=MIN_KEYWORD_MATCHES
)

dense_precision, dense_precision_detail = evaluate_precision_at_k_page_keyword(
    eval_df_v4,
    retrieve_func=retrieve_clean_dense_wrapper,
    k=K,
    tolerance=TOLERANCE,
    min_keyword_matches=MIN_KEYWORD_MATCHES
)

dense_mrr, dense_mrr_detail = evaluate_mrr_at_k_page_keyword(
    eval_df_v4,
    retrieve_func=retrieve_clean_dense_wrapper,
    k=K,
    tolerance=TOLERANCE,
    min_keyword_matches=MIN_KEYWORD_MATCHES
)

# Reranked retrieval evaluation
rerank_recall, rerank_recall_detail = evaluate_recall_at_k_page_keyword(
    eval_df_v4,
    retrieve_func=retrieve_clean_rerank_wrapper,
    k=K,
    tolerance=TOLERANCE,
    min_keyword_matches=MIN_KEYWORD_MATCHES
)

rerank_precision, rerank_precision_detail = evaluate_precision_at_k_page_keyword(
    eval_df_v4,
    retrieve_func=retrieve_clean_rerank_wrapper,
    k=K,
    tolerance=TOLERANCE,
    min_keyword_matches=MIN_KEYWORD_MATCHES
)

rerank_mrr, rerank_mrr_detail = evaluate_mrr_at_k_page_keyword(
    eval_df_v4,
    retrieve_func=retrieve_clean_rerank_wrapper,
    k=K,
    tolerance=TOLERANCE,
    min_keyword_matches=MIN_KEYWORD_MATCHES
)

summary_results = pd.DataFrame([
    {
        "method": "Clean Dense Retrieval",
        "Recall@5": dense_recall,
        "Precision@5": dense_precision,
        "MRR@5": dense_mrr
    },
    {
        "method": "Clean Dense Retrieval + Cross-Encoder Reranking",
        "Recall@5": rerank_recall,
        "Precision@5": rerank_precision,
        "MRR@5": rerank_mrr
    }
])

display(summary_results)

summary_results.to_csv(DATA_PROCESSED_DIR / "evaluation_summary_v4.csv", index=False)

In [ ]:
# Per-query comparison
dense_detail_for_compare = dense_mrr_detail[
    ["question", "first_hit_rank", "reciprocal_rank", "hit_page", "matched_keywords", "hit_preview"]
].copy()

rerank_detail_for_compare = rerank_mrr_detail[
    ["question", "first_hit_rank", "reciprocal_rank", "hit_page", "matched_keywords", "hit_preview"]
].copy()

comparison_v4 = dense_detail_for_compare.merge(
    rerank_detail_for_compare,
    on="question",
    suffixes=("_dense", "_rerank")
)

comparison_v4["mrr_change"] = (
    comparison_v4["reciprocal_rank_rerank"] -
    comparison_v4["reciprocal_rank_dense"]
)

display(comparison_v4)

comparison_v4.to_csv(DATA_PROCESSED_DIR / "evaluation_comparison_v4.csv", index=False)

## 9. Evidence Context Construction

This section turns reranked chunks into a compact evidence context for answer generation.

In [ ]:
def build_evidence_context(results, max_chars=4500):
    """
    Build a compact evidence context from retrieved chunks.

    Each source includes page and chunk metadata, which helps the answer generator cite evidence.
    """
    context_parts = []
    total_chars = 0

    for i, r in enumerate(results):
        source_header = f"[Source {i+1}] Page {r['page']}, Chunk {r['chunk_id']}"
        source_text = r["text"].strip()
        source_block = f"{source_header}\n{source_text}"

        if total_chars + len(source_block) > max_chars:
            break

        context_parts.append(source_block)
        total_chars += len(source_block)

    return "\n\n".join(context_parts)


def build_rag_prompt(query, results):
    """
    Build an evidence-grounded RAG prompt.
    """
    context = build_evidence_context(results)

    prompt = f"""
You are an AI assistant answering questions about a long AI report.

Use only the provided sources to answer the question.
Do not use outside knowledge.
If the sources do not contain enough information, say:
"The provided context is insufficient to answer this question."

When possible, cite evidence using the source number and page number, such as:
(Source 1, Page 248).

Question:
{query}

Sources:
{context}

Answer:
""".strip()

    return prompt


query = "How has AI investment changed in recent years?"
results = retrieve_with_rerank_clean(query, first_k=FIRST_K, top_k=TOP_K)
prompt = build_rag_prompt(query, results)

print(prompt[:4000])

## 10. Optional LLM Answer Generation

This notebook supports two answer modes:

1. **OpenAI API mode** if `OPENAI_API_KEY` is available in your environment.
2. **Extractive fallback mode** if no API key is available.

The fallback mode does not generate a fluent LLM-style answer, but it still returns grounded evidence from retrieved sources. This keeps the notebook runnable without paid API access.

In [ ]:
def generate_answer_with_openai(prompt, model="gpt-4o-mini"):
    """
    Optional OpenAI-based answer generation.

    Requires:
        pip install openai
        environment variable OPENAI_API_KEY

    Returns:
        str or None
    """
    api_key = os.getenv("OPENAI_API_KEY")

    if not api_key:
        return None

    try:
        from openai import OpenAI

        client = OpenAI(api_key=api_key)

        response = client.chat.completions.create(
            model=model,
            messages=[
                {
                    "role": "system",
                    "content": "You answer questions strictly using the provided context and cite sources."
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            temperature=0.2
        )

        return response.choices[0].message.content

    except Exception as e:
        print("OpenAI generation failed:", e)
        return None

In [ ]:
def extractive_answer_from_evidence(query, results, max_sources=3, max_chars_per_source=700):
    """
    Fallback answer mode without using an external LLM.
    It returns the most relevant evidence snippets with page-level citations.
    """
    lines = []
    lines.append(f"Question: {query}")
    lines.append("")
    lines.append("Evidence-based answer draft:")
    lines.append(
        "The retrieved evidence below contains the most relevant passages for answering the question. "
        "Use these cited sources to write a final natural-language answer."
    )
    lines.append("")

    for i, r in enumerate(results[:max_sources]):
        snippet = r["text"][:max_chars_per_source].replace("\n", " ").strip()
        lines.append(f"[Source {i+1}] Page {r['page']}, Chunk {r['chunk_id']}")
        lines.append(snippet)
        lines.append("")

    return "\n".join(lines)


def answer_question(query, first_k=20, top_k=5, use_openai=True):
    """
    End-to-end RAG answer function:
    query -> retrieve -> rerank -> build prompt -> generate answer or fallback evidence output
    """
    results = retrieve_with_rerank_clean(
        query=query,
        first_k=first_k,
        top_k=top_k
    )

    prompt = build_rag_prompt(query, results)

    generated_answer = None

    if use_openai:
        generated_answer = generate_answer_with_openai(prompt)

    if generated_answer is None:
        generated_answer = extractive_answer_from_evidence(query, results)

    return {
        "query": query,
        "answer": generated_answer,
        "retrieved_results": results,
        "prompt": prompt
    }

In [ ]:
# Test Version 4 answer generation
query = "How has AI investment changed in recent years?"

rag_output = answer_question(
    query=query,
    first_k=FIRST_K,
    top_k=TOP_K,
    use_openai=True  # If no API key is set, fallback evidence mode will be used
)

print(rag_output["answer"])

## 11. Batch QA Demo

This section runs the Version 4 RAG system on multiple evaluation questions and saves the outputs.

In [ ]:
demo_questions = [
    "How has AI investment changed in recent years?",
    "What does the report say about the cost of AI inference?",
    "How is AI being used in healthcare?",
    "What risks or challenges are associated with AI adoption?",
    "What does the report say about AI regulation?"
]

qa_rows = []

for q in demo_questions:
    output = answer_question(
        query=q,
        first_k=FIRST_K,
        top_k=TOP_K,
        use_openai=True
    )

    top_sources = [
        {
            "rank": r["rank"],
            "page": r["page"],
            "chunk_id": r["chunk_id"],
            "cosine_score": r.get("cosine_score"),
            "rerank_score": r.get("rerank_score")
        }
        for r in output["retrieved_results"]
    ]

    qa_rows.append({
        "question": q,
        "answer": output["answer"],
        "top_sources": json.dumps(top_sources)
    })

qa_demo_df = pd.DataFrame(qa_rows)
display(qa_demo_df)

qa_demo_df.to_csv(DATA_PROCESSED_DIR / "qa_demo_outputs_v4.csv", index=False)
print("Saved:", DATA_PROCESSED_DIR / "qa_demo_outputs_v4.csv")

## 12. Inspect Retrieval and Answer for One Query

Use this helper for debugging and qualitative analysis.

In [ ]:
def inspect_rag_answer(query, first_k=20, top_k=5):
    output = answer_question(
        query=query,
        first_k=first_k,
        top_k=top_k,
        use_openai=True
    )

    print("=" * 120)
    print("QUESTION")
    print("=" * 120)
    print(query)

    print("\n" + "=" * 120)
    print("ANSWER")
    print("=" * 120)
    print(output["answer"])

    print("\n" + "=" * 120)
    print("RETRIEVED SOURCES")
    print("=" * 120)

    for r in output["retrieved_results"]:
        print(
            f"Rank: {r['rank']} | Page: {r['page']} | Chunk: {r['chunk_id']} | "
            f"Cosine: {r['cosine_score']:.4f} | Rerank: {r['rerank_score']:.4f}"
        )
        print(r["text"][:1000])
        print("-" * 120)

    return output


# Example:
# inspect_rag_answer("What does the report say about the cost of AI inference?")

## 13. Summary

Version 4 extends the retrieval pipeline into a complete RAG-style document QA system.

### Key Improvements

1. **Cleaned retrieval corpus**  
   Removed Table of Contents chunks, repeated headers, standalone page numbers, and very short chunks.

2. **Reranked retrieval**  
   Used dense FAISS retrieval for candidate generation and a cross-encoder reranker for ranking quality.

3. **Page-aware evaluation**  
   Evaluated retrieval using manually labeled expected pages and evidence keywords, instead of keyword-only matching.

4. **Evidence-grounded answer generation**  
   Built context blocks with page-level source metadata and generated answers using an LLM when available.

5. **Runnable fallback mode**  
   If no OpenAI API key is available, the system returns cited evidence snippets, keeping the notebook reproducible without external API calls.